In [ ]:
# Exercise 9.13 Schrodinger
from numpy import empty, arange, linspace, exp, real, full
import matplotlib.pyplot as plt
import matplotlib.animation as animation
#from banded import banded
from matplotlib import rc,rcParams
from IPython.display import HTML
from numpy import copy

rcParams['animation.embed_limit'] = 2**128

def banded(Aa,va,up,down):

    # Copy the inputs and determine the size of the system
    A = copy(Aa)
    v = copy(va)
    N = len(v)

    # Gaussian elimination
    for m in range(N):

        # Normalization factor
        div = A[up,m]

        # Update the vector first
        v[m] /= div
        for k in range(1,down+1):
            if m+k<N:
                v[m+k] -= A[up+k,m]*v[m]

        # Now normalize the pivot row of A and subtract from lower ones
        for i in range(up):
            j = m + up - i
            if j<N:
                A[i,j] /= div
                for k in range(1,down+1):
                    A[i+k,j] -= A[up+k,m]*A[i,j]

    # Backsubstitution
    for m in range(N-2,-1,-1):
        for i in range(up):
            j = m + up - i
            if j<N:
                v[m] -= A[i,j]*v[j]

    return v

#constants
L = 3.0e-9
N = 5000
a = L/N
m = 9.109e-31
hbar = 1.055e-34
x0 = L/2
sigma = 1.0e-10
kappa = 5.0e10

yscale = 1.5e-9
h = 1e-19

C = 1j*hbar/(4*m*a*a)
a1 = 1 + 2*h*C
a2 = -h*C
b1 = 1 - 2*h*C
b2 = h*C

# Create the initial arrays of x and psi values
x = linspace(0,L,N+1)
psi = exp(-(x-x0)**2/(2*sigma**2))*exp(1j*kappa*x)
psi[0] = psi[N] = 0
# Create the tridiagonal array A
A = empty([3,N-1],complex)
A[0,:] = a2
A[1,:] = a1
A[2,:] = a2
lines = []

fig = plt.figure()
ax = plt.axes(xlim=(0, L), ylim=(0,1.1*yscale))
line0 = ax.plot([], [], lw=3,color="red",label="Real piece of Psi")
line1 = ax.plot([], [], lw=3,color="blue",label="Magnitude of Psi")
lines.append(line0[0])
lines.append(line1[0])
plt.legend()

def init():
  for line in lines:
    line.set_data([], [])
  return lines


def animate(i):
    ## main loop for the Crank-Nicolson method
    v = b2*psi[0:N-1] + b1*psi[1:N] + b2*psi[2:N+1]
    psi[1:N] = banded(A,v,1,1)
    ys0 = yscale*real(psi)
    ys1 = yscale*abs(psi)
    lines[0].set_data(x,ys0)
    lines[1].set_data(x,ys1)
    return lines


ani = animation.FuncAnimation(fig, animate, interval=1, frames = 10000, init_func=init,blit=True)
# Note: below is the part which makes it work on Colab
rc('animation', html='jshtml')
ani

###plt.show()

